# Constrained and Guided Generation Tutorial

## Overview

This tutorial explores the concepts of constrained and guided generation in the context of large language models. We'll focus on techniques to set up constraints for model outputs and implement rule-based generation using OpenAI's GPT models and the LangChain library.

## Motivation

While large language models are powerful tools for generating text, they sometimes produce outputs that are too open-ended or lack specific desired characteristics. Constrained and guided generation techniques allow us to exert more control over the model's outputs, making them more suitable for specific tasks or adhering to certain rules and formats.

## Key Components

1. Setting up constraints for model outputs
2. Implementing rule-based generation
3. Using LangChain's PromptTemplate for structured prompts
4. Leveraging OpenAI's GPT models for text generation

## Method Details

We'll use a combination of prompt engineering techniques and LangChain's utilities to implement constrained and guided generation:

1. We'll start by setting up the environment and importing necessary libraries.
2. We'll create structured prompts using LangChain's PromptTemplate to guide the model's output.
3. We'll implement constraints by specifying rules and formats in our prompts.
4. We'll use OpenAI's GPT model to generate text based on our constrained prompts.
5. We'll explore different techniques for rule-based generation, including output parsing and regex-based validation.

## Conclusion

By the end of this tutorial, you'll have a solid understanding of how to implement constrained and guided generation techniques. These skills will enable you to create more controlled and specific outputs from large language models, making them more suitable for a wide range of applications where precise and rule-adherent text generation is required.

## Setup

First, let's import the necessary libraries and set up our environment.

In [2]:
import os
import re

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

from dotenv import load_dotenv
load_dotenv()

# Initialize the language model (Gemini)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Function to display model outputs
def display_output(output):
    """Display the model's output in a formatted manner."""
    print("Model Output:")
    print("-" * 40)
    print(output)
    print("-" * 40)
    print()

## Setting Up Constraints for Model Outputs

Let's start by creating a constrained prompt that generates a product description with specific requirements.

In [3]:
constrained_prompt = PromptTemplate(
    input_variables=["product", "target_audience", "tone", "word_limit"],
    template="""Create a product description for {product} targeted at {target_audience}.
    Use a {tone} tone and keep it under {word_limit} words.
    The description should include:
    1. A catchy headline
    2. Three key features
    3. A call to action
    
    Product Description:
    """
)

# Generate the constrained output
input_variables = {
    "product": "smart water bottle",
    "target_audience": "health-conscious millennials",
    "tone": "casual and friendly",
    "word_limit": "75"
}

chain = constrained_prompt | llm
output = chain.invoke(input_variables).content
display_output(output)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Model Output:
----------------------------------------
**Sip Smart, Feel Great.**

Meet your new hydration BFF! This smart water bottle automatically tracks your water intake, glows to remind you when to drink, and syncs with your phone for personalized goals. Stay on top of your wellness effortlessly. Grab yours today and glow from within!
----------------------------------------



## Implementing Rule-Based Generation

Now, let's implement a rule-based generation system for creating structured job postings.

In [4]:
job_posting_prompt = PromptTemplate(
    input_variables=["job_title", "company", "location", "experience"],
    template="""Create a job posting for a {job_title} position at {company} in {location}.
    The candidate should have {experience} years of experience.
    Follow these rules:
    1. Start with a brief company description (2 sentences)
    2. List 5 key responsibilities, each starting with an action verb
    3. List 5 required qualifications, each in a single sentence
    4. End with a standardized equal opportunity statement
    
    Format the output as follows:
    COMPANY: [Company Description]
    
    RESPONSIBILITIES:
    - [Responsibility 1]
    - [Responsibility 2]
    - [Responsibility 3]
    - [Responsibility 4]
    - [Responsibility 5]
    
    QUALIFICATIONS:
    - [Qualification 1]
    - [Qualification 2]
    - [Qualification 3]
    - [Qualification 4]
    - [Qualification 5]
    
    EEO: [Equal Opportunity Statement]
    """
)

# Generate the rule-based output
input_variables = {
    "job_title": "Senior Software Engineer",
    "company": "TechInnovate Solutions",
    "location": "San Francisco, CA",
    "experience": "5+"
}

chain = job_posting_prompt | llm
output = chain.invoke(input_variables).content
display_output(output)

Model Output:
----------------------------------------
COMPANY: TechInnovate Solutions is a pioneering technology company based in the vibrant San Francisco Bay Area. We are dedicated to building cutting-edge software products that redefine industries and empower our users.

RESPONSIBILITIES:
- Design and implement scalable, high-performance software systems and architectures.
- Develop robust, maintainable, and efficient code across the full software development lifecycle.
- Lead technical discussions, provide mentorship to junior engineers, and champion engineering best practices.
- Collaborate cross-functionally with product managers, designers, and other engineering teams to define and deliver features.
- Optimize existing systems for improved performance, reliability, and security.

QUALIFICATIONS:
- You possess 5+ years of professional experience in software development.
- You have strong proficiency in at least one modern programming language such as Python, Java, Go, or C#.
- Y

## Using Regex Parser for Structured Output

Let's use a regex parser to ensure our output adheres to a specific structure.

In [5]:
# Define a regex pattern for structured output
regex_pattern = r"COMPANY:\s*([\s\S]*?)\n\s*RESPONSIBILITIES:\s*([\s\S]*?)\n\s*QUALIFICATIONS:\s*([\s\S]*?)\n\s*EEO:\s*([\s\S]*)"
output_keys = ["company_description", "responsibilities", "qualifications", "eeo_statement"]
# This regex pattern captures the company description, responsibilities, qualifications, and EEO statement from the output text.

# Create a new prompt template that includes the parser instructions
parsed_job_posting_prompt = PromptTemplate(
    input_variables=["job_title", "company", "location", "experience"],
    template="""Create a job posting for a {job_title} position at {company} in {location}.
    The candidate should have {experience} years of experience.
    Follow these rules:
    1. Start with a brief company description (2 sentences)
    2. List 5 key responsibilities, each starting with an action verb
    3. List 5 required qualifications, each in a single sentence
    4. End with a standardized equal opportunity statement
    
    Format the output EXACTLY as follows:
    COMPANY: [Company Description]
    
    RESPONSIBILITIES:
    - [Responsibility 1]
    - [Responsibility 2]
    - [Responsibility 3]
    - [Responsibility 4]
    - [Responsibility 5]
    
    QUALIFICATIONS:
    - [Qualification 1]
    - [Qualification 2]
    - [Qualification 3]
    - [Qualification 4]
    - [Qualification 5]
    
    EEO: [Equal Opportunity Statement]
    """
)

def clean_output(output):
    for key, value in output.items():
        if isinstance(value, str):
            # Remove leading/trailing whitespace and normalize newlines
            output[key] = re.sub(r'\n\s*', '\n', value.strip())
    return output

# Generate the parsed output
chain = parsed_job_posting_prompt | llm
raw_output = chain.invoke(input_variables).content

# Parse and clean the output
match = re.search(regex_pattern, raw_output)
parsed_output = dict(zip(output_keys, match.groups())) if match else {}
cleaned_output = clean_output(parsed_output)

# Display the parsed output
print("Parsed Output:")
for key, value in cleaned_output.items():
    print(f"{key.upper()}:")
    print(value)
    print()

Parsed Output:
COMPANY_DESCRIPTION:
TechInnovate Solutions is a pioneering technology company based in San Francisco, CA, dedicated to building innovative software products that solve complex real-world problems. We foster a collaborative and dynamic environment where creativity and technical excellence drive our success.

RESPONSIBILITIES:
- Design and develop scalable, high-performance software solutions across our product suite.
- Lead technical discussions, mentor junior engineers, and contribute to architectural decisions.
- Collaborate with product managers, designers, and other engineers to define, scope, and deliver features.
- Write clean, maintainable, and well-tested code following best practices and coding standards.
- Troubleshoot and debug complex issues, ensuring the reliability and stability of our systems.

QUALIFICATIONS:
- You have 5+ years of professional experience in software development.
- You possess strong proficiency in at least one modern programming language

## Implementing Additional Constraints

Let's create a more complex constrained generation task: generating a product review with specific criteria.

In [6]:
review_prompt = PromptTemplate(
    input_variables=["product", "rating", "pros", "cons", "word_limit"],
    template="""Write a product review for {product} with the following constraints:
    1. The review should have a {rating}-star rating (out of 5)
    2. Include exactly {pros} pros and {cons} cons
    3. Use between 2 and 3 sentences for each pro and con
    4. The entire review should be under {word_limit} words
    5. End with a one-sentence recommendation
    
    Format the review as follows:
    Rating: [X] out of 5 stars
    
    Pros:
    1. [Pro 1]
    2. [Pro 2]
    ...
    
    Cons:
    1. [Con 1]
    2. [Con 2]
    ...
    
    Recommendation: [One-sentence recommendation]
    """
)

# Generate the constrained review
input_variables = {
    "product": "Smartphone X",
    "rating": "4",
    "pros": "3",
    "cons": "2",
    "word_limit": "200"
}

chain = review_prompt | llm
output = chain.invoke(input_variables).content
display_output(output)

Model Output:
----------------------------------------
Rating: 4 out of 5 stars

Pros:
1.  Smartphone X truly excels in battery endurance. I consistently get a full day of heavy use, sometimes even stretching into the next morning without needing a charge.
2.  The camera on Smartphone X captures stunning photos in good lighting conditions. Colors are vibrant, and details are sharp, making it great for everyday snapshots.
3.  Overall performance is remarkably smooth for its price point. Apps launch quickly, and multitasking feels effortless, handling most tasks without a hitch.

Cons:
1.  While the display is vibrant indoors, its outdoor visibility can be a struggle. In direct sunlight, readability significantly decreases, requiring you to find shade.
2.  The single bottom-firing speaker is somewhat underwhelming. Audio lacks depth and can sound tinny, especially at higher volumes.

Recommendation: For its price, Smartphone X offers an excellent balance of features and performance, maki

![](https://europe-west1-prompt-eng-views-tracker.cloudfunctions.net/prompt-eng-tracker?notebook=all-prompt-engineering-techniques--constrained-guided-generation)